In [ ]:
-- 1) baseline: месяц ДО начала кампаний
with client_baseline as (
    select
        cc.client_id,
        cc.campaigns_cnt,
        coalesce(sum(ch.summ_discounted), 0) as baseline_spend
    from _ cc
    left join _ ch
        on ch.contact_id = cc.client_id
       and ch.datetime >= date '{date_start}' - interval '1 month'
       and ch.datetime <  date '{date_start}'
       and ch.operation_type_id = 1
       and ch.summ_discounted > 0
    group by cc.client_id, cc.campaigns_cnt
),

-- 2) траты по месяцам в периоде кампаний
client_month_spend as (
    select
        cc.client_id,
        cc.campaigns_cnt,
        date_trunc('month', ch.datetime)::date as month_dt,
        sum(ch.summ_discounted) as month_spend
    from _ cc
    left join dm.cheque ch
        on ch.contact_id = cc.client_id
       and ch.datetime >= date '{date_start}'
       and ch.datetime <  date '{date_end}' + interval '1 day'
       and ch.operation_type_id = 1
       and ch.summ_discounted > 0
    group by cc.client_id, cc.campaigns_cnt, month_dt
),

-- 3) дельта на уровне клиента
client_delta as (
    select
        ms.client_id,
        ms.campaigns_cnt,
        ms.month_dt,
        coalesce(ms.month_spend, 0) as month_spend,
        cb.baseline_spend,
        coalesce(ms.month_spend, 0) - cb.baseline_spend as delta_spend
    from client_month_spend ms
    join client_baseline cb
        on ms.client_id = cb.client_id
),

-- 4) агрегирование по когортам
cohort_delta as (
    select
        campaigns_cnt,
        month_dt,
        avg(delta_spend) as avg_delta_spend,                 -- основная метрика
        sum(month_spend) as total_spend,                     -- контекст
        avg(month_spend) as avg_spend_per_client             -- контекст
    from client_delta
    group by campaigns_cnt, month_dt
)

select
    campaigns_cnt,
    month_dt,
    round(avg_delta_spend, 2) as avg_delta_spend,
    round(avg_spend_per_client, 2) as avg_spend_per_client,
    total_spend
from cohort_delta
order by campaigns_cnt, month_dt;

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

df_delta['month_dt'] = pd.to_datetime(df_delta['month_dt'])

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Delta spend
for campaigns_cnt, group in df_delta.groupby('campaigns_cnt'):
    group = group.sort_values('month_dt')

    axes[0].plot(
        group['month_dt'],
        group['avg_delta_spend'],
        marker='o',
        label=f'{campaigns_cnt}'
    )

axes[0].axhline(0, linestyle='--')

axes[0].set_title('Δ траты относительно baseline')
axes[0].set_xlabel('Месяц')
axes[0].set_ylabel('Δ spend')

axes[0].xaxis.set_major_locator(mdates.MonthLocator())
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

axes[0].grid(alpha=0.3)

# 2. Average spend per client
for campaigns_cnt, group in df_delta.groupby('campaigns_cnt'):
    group = group.sort_values('month_dt')

    axes[1].plot(
        group['month_dt'],
        group['avg_spend_per_client'],
        marker='o',
        label=f'{campaigns_cnt}'
    )

axes[1].set_title('Средние траты на клиента')
axes[1].set_xlabel('Месяц')
axes[1].set_ylabel('Avg spend per client')

axes[1].xaxis.set_major_locator(mdates.MonthLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

axes[1].grid(alpha=0.3)

handles, labels = axes[1].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    title='Кол-во кампаний',
    bbox_to_anchor=(1.02, 0.9)
)

plt.tight_layout()
plt.show()